<a href="https://colab.research.google.com/github/suprajaa4/Bioinfo/blob/main/bioimaging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Identification of Active-Site Pockets Using Morphological Hole Filling

Maha El Khalil, Suprajaa Venkatasubramanian & Tanushka Mantri

---


## Project queation
Can simple image-processing morphology (hole filling + segmentation) applied to image-like representations of protein surfaces automatically detect and quantify surface pockets that correspond to biologically meaningful active sites?

## Why this matters
Active sites are often located in surface depressions (“pockets”). Many pocket-detection tools exist, but this project tests whether a minimal, explainable pipeline from basic morphology can recover the same kind of features.

## Core idea
If we treat a protein’s surface/occupancy grid as an image, then pockets behave like “holes” or enclosed cavities in that image. Morphological hole filling can highlight those regions, and segmentation can separate individual pockets.

## Method overview
1. Convert PDB coordinates into 2D image-like grids representing protein occupancy/surface.
2. Apply morphological operations (hole filling and cleanup) to isolate pocket-like regions.
3. Segment the resulting pocket candidates and assign object labels.

## Quantification
- Pocket count per protein
- Pocket size (area/volume; pixel/voxel units)
- Pocket shape descriptors (e.g., compactness, elongation)

## Expected Pattern
Enzymes with known active sites (lysozyme, trypsin, carbonic anhydrase) are expected to show fewer but more prominent/consistent pockets than non-enzymatic globular proteins (myoglobin, hemoglobin subunit, serum albumin fragment), which should display smaller or less distinctive pocket-like regions.

### Step 1: Data Acquisition & Image Construction
*Maha El Khalil*


---



Proteins were chosen to represent enzymes with known binding pockets and non-enzymatic globular proteins for comparison.

Enzymes (with known active sites):
 - Lysozyme (e.g. PDB: 1LYZ)
 - Trypsin (2PTN)
 - Carbonic anhydrase (1CA2)

Non-enzymes / structural proteins:
 - Myoglobin (1MBN)
 - Hemoglobin subunit (1A3N)
 - Serum albumin fragment

1. load PDB + extract a clean coordinate table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

def parse_pdb_atoms(pdb_file, chain_id="A", atom_name="CA"):
    """
    Minimal PDB parser for Legacy PDB format.
    Extracts ATOM records for a given chain and atom type (default: CA).
    """
    rows = []
    pdb_file = Path(pdb_file)

    with pdb_file.open("r") as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue

            # PDB fixed-width columns (Legacy PDB)
            # atom name: cols 13-16, chain: col 22, resSeq: cols 23-26
            atom = line[12:16].strip()
            chain = line[21].strip()
            res_name = line[17:20].strip()
            res_seq = int(line[22:26])
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])

            if chain == chain_id and atom == atom_name:
                rows.append({
                    "res_name": res_name,
                    "res_seq": res_seq,
                    "atom": atom,
                    "x": x, "y": y, "z": z
                })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No atoms found for chain={chain_id}, atom={atom_name}. "
                         f"Check chain IDs / file content.")
    return df

pdb_dir = Path("/content/drive/MyDrive/BioimagingProject/data/pdb/")
pdb_files = sorted(pdb_dir.glob("*.pdb"))

all_ca = {}

for f in pdb_files:
    pdb_id = f.stem  # e.g., "1LYZ"
    df = parse_pdb_atoms(f, chain_id="A", atom_name="CA")
    all_ca[pdb_id] = df

    print(f"\n=== {pdb_id} ===")
    print("Number of CA atoms:", len(df))
    print("Residue range:", df["res_seq"].min(), "-", df["res_seq"].max())
    print(df[["x","y","z"]].agg(["min","max"]))

# for reproducibility
out_dir = Path("/content/drive/MyDrive/BioimagingProject/derived/ca_tables")
out_dir.mkdir(parents=True, exist_ok=True)

for pdb_id, df in all_ca.items():
    df.to_csv(out_dir / f"{pdb_id}_CA.csv", index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== 1A3N ===
Number of CA atoms: 141
Residue range: 1 - 141
          x       y       z
min -13.992  -3.583   3.918
max  24.863  28.073  39.076

=== 1AO6 ===
Number of CA atoms: 578
Residue range: 5 - 582
          x       y       z
min  -0.830   2.440 -14.220
max  66.266  57.826  57.996

=== 1CA2 ===
Number of CA atoms: 256
Residue range: 4 - 260
          x       y       z
min -27.731 -19.796  -8.336
max  13.457  19.905  38.301

=== 1LYZ ===
Number of CA atoms: 129
Residue range: 1 - 129
          x       y       z
min -15.952   7.910   0.695
max  18.589  37.639  36.259

=== 1MBN ===
Number of CA atoms: 153
Residue range: 1 - 153
        x     y     z
min  -4.0   2.7  -7.6
max  34.2  35.5  23.6

=== 2PTN ===
Number of CA atoms: 223
Residue range: 16 - 245
          x       y       z
min -15.243  -9.411   1.289
max  22.243  24.156  44.243


2. PCA alignment (make orientations comparable)
Proteins are stored with arbitrary rotation in PDB files. PCA rotates the coordinate cloud into a consistent frame:
  - PC1 & PC2 become image axes (x/y)
  - PC3 becomes depth (pixel intensity)

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

def pca_align_xyz(df):
    coords = df[["x", "y", "z"]].to_numpy(float)

    # center the protein at the origin
    coords_centered = coords - coords.mean(axis=0, keepdims=True)

    # rotate into PCA coordinate system
    coords_pca = PCA(n_components=3).fit_transform(coords_centered)

    # return u,v as image coords and d as depth
    u = coords_pca[:, 0]
    v = coords_pca[:, 1]
    d = coords_pca[:, 2]
    return u, v, d


3. convert continuous coordinates into a 2D grid of pixels



In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter

def rasterize_depth(u, v, d, img_size=256, padding=0.05, sigma_px=3.0):
    # map u,v to pixels
    umin, umax = u.min(), u.max()
    vmin, vmax = v.min(), v.max()
    ur, vr = umax - umin, vmax - vmin

    umin -= padding * ur; umax += padding * ur
    vmin -= padding * vr; vmax += padding * vr
    ur = max(umax - umin, 1e-9)
    vr = max(vmax - vmin, 1e-9)

    xpix = ((u - umin) / ur * (img_size - 1)).astype(int)
    ypix = ((v - vmin) / vr * (img_size - 1)).astype(int)
    xpix = np.clip(xpix, 0, img_size - 1)
    ypix = np.clip(ypix, 0, img_size - 1)

    depth_sum = np.zeros((img_size, img_size), dtype=float)
    weight_sum = np.zeros((img_size, img_size), dtype=float)

    for x, y, val in zip(xpix, ypix, d):
        depth_sum[y, x] += val
        weight_sum[y, x] += 1.0

    depth_sum_f = gaussian_filter(depth_sum, sigma=sigma_px)
    weight_sum_f = gaussian_filter(weight_sum, sigma=sigma_px)

    depth = depth_sum_f / (weight_sum_f + 1e-8)
    return depth, weight_sum_f


4. normalize depth to 0-255 (grayscale image)

In [ ]:
def depth_to_uint8_masked(depth, mask, clip_percentiles=(2, 98)):
    vals = depth[mask]
    lo, hi = np.percentile(vals, clip_percentiles)
    if hi <= lo:
        lo, hi = float(vals.min()), float(vals.max() + 1e-9)

    norm = np.zeros_like(depth, dtype=float)
    norm[mask] = np.clip((depth[mask] - lo) / (hi - lo), 0, 1)

    img_u8 = (norm * 255).astype(np.uint8)
    return img_u8


5. run full pipeline on each protein resulting in 2 images for each protein in image folder:
  - _mask.png → where the protein is
  - \_depth.png → depth signal inside the protein region

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import binary_closing, binary_fill_holes, label

ca_dir = Path("/content/drive/MyDrive/BioimagingProject/derived/ca_tables")
img_dir = Path("/content/drive/MyDrive/BioimagingProject/images/depth_maps")
img_dir.mkdir(parents=True, exist_ok=True)

def keep_largest_component(mask):
    labeled, n = label(mask)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    return labeled == sizes.argmax()

csv_files = sorted(ca_dir.glob("*_CA.csv"))

for csv_file in csv_files:
    pdb_id = csv_file.stem.replace("_CA", "")
    df = pd.read_csv(csv_file)

    u, v, d = pca_align_xyz(df)
    depth, w = rasterize_depth(u, v, d, img_size=256, sigma_px=4.0)

    mask = w > (0.02 * w.max())
    mask = binary_closing(mask, iterations=3)
    mask = binary_fill_holes(mask)
    mask = keep_largest_component(mask)

    img_u8 = depth_to_uint8_masked(depth, mask, clip_percentiles=(2, 98))

    Image.fromarray(img_u8, mode="L").save(img_dir / f"{pdb_id}_depth.png")
    Image.fromarray((mask.astype(np.uint8) * 255), mode="L").save(img_dir / f"{pdb_id}_mask.png")

print("Saved", len(csv_files), "depth maps + masks to:", img_dir)


/tmp/ipython-input-3304117779.py:34: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(img_u8, mode="L").save(img_dir / f"{pdb_id}_depth.png")
/tmp/ipython-input-3304117779.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray((mask.astype(np.uint8) * 255), mode="L").save(img_dir / f"{pdb_id}_mask.png")
/tmp/ipython-input-3304117779.py:34: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(img_u8, mode="L").save(img_dir / f"{pdb_id}_depth.png")
/tmp/ipython-input-3304117779.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray((mask.astype(np.uint8) * 255), mode="L").save(img_dir / f"{pdb_id}_mask.png")
/tmp/ipython-input-3304117779.py:34: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (20

Saved 6 depth maps + masks to: /content/drive/MyDrive/BioimagingProject/images/depth_maps


/tmp/ipython-input-3304117779.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray((mask.astype(np.uint8) * 255), mode="L").save(img_dir / f"{pdb_id}_mask.png")


### Step 2: Morphological Identification and Characterization
*Suprajaa Venkatasubramanian*


---

The morphology pipeline's goal is to identify potential surface pockets as nearby areas that match small depressions in the surface picture. Then uses common morphological descriptors to quantify each potential pocket.



Required: both input PNGs for each protein i.e,

 1.grayscale depth map (pdb.id_depth.png) representing a 2D projection of the protein surface (values scaled to 0 to 255),

 2.binary protein footprint mask (pdb.id_mask.png) defining pixels of the protein projection.


In [ ]:
#all required packages.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from skimage.io import imread
from skimage.morphology import disk, diamond, square, dilation, erosion
from skimage.filters import threshold_otsu
from skimage.measure import label as sk_label, regionprops_table
from skimage.segmentation import find_boundaries
from scipy.ndimage import binary_fill_holes, binary_closing, label as ndi_label, distance_transform_edt

In [ ]:
#setting output path
depth_dir = Path("/content/drive/MyDrive/BioimagingProject/images/depth_maps")
mask_dir  = Path("/content/drive/MyDrive/BioimagingProject/images/depth_maps")

out_path = Path("/content/drive/MyDrive/BioimagingProject/morphology")
out_path.mkdir(parents=True, exist_ok=True)

### 1. Grayscale closing-

It is a classical morphological tool for removing small dark concavities while maintaining overall object shape.

First dilation raises local minima (fills depressions).

Then erosion restores the overall shape without reintroducing small valleys.

Finally: pockets smaller than the structuring element (SE) are filled.

I_closed = (I ⊕ B)⊖ B
Where I= normalized surface image
⊕= grayscale dilation
⊖= grayscale erosion
B= structuring element

In [ ]:
def gray_closing(img, se):
    return erosion(dilation(img, footprint=se), footprint=se)

### 2.Pocket detection from a surface image (I) and footprint mask

Converting the filled-vs-original difference into a “pocket depth map,” threshold it to obtain pocket regions, fill holes, and label connected components.

D (Pocket Depth Map) = max(I_closed- I,0)

Operations include:

1. Subtraction (I_closed − I_original) isolates exactly what the closing operation filled, i.e., positive values indicate stronger depressions.

2. Thresholding (Otsu) provides an objective separation between strong pocket responses and background variations in the difference image.

3. Hole filling ensures pocket regions are solid blobs rather than fragmented rings.

4. Connected component labeling identifies individual pockets and enables region-level measurements.

In [ ]:
def detect_pockets(I, mask, se, min_area=30):
    closed = gray_closing(I, se)
    pdepth = np.clip(closed - I, 0, None) #subtraction
    pdepth[~mask] = 0
    #outside mask set to 0
    th = threshold_otsu(pdepth[mask]) if mask.any() else 0
    pbin = (pdepth > th) & mask
    #filling holes inside pocket candidates so each pocket = solid region
    pbin = binary_fill_holes(pbin)

    lbl = sk_label(pbin, connectivity=2)
    #reduce noise
    if lbl.max() > 0:
        keep = np.zeros_like(lbl, bool)
        for rid in range(1, lbl.max()+1):
            if np.sum(lbl == rid) >= min_area:
                keep |= (lbl == rid)
        pbin = keep
        lbl = sk_label(pbin, connectivity=2)

    return closed, pdepth, pbin, lbl, float(th)

### 3. Structuring elements for multi-scale pocket detection

Testing multiple structuring element shapes and sizes because pocket-like depressions can vary in geometry and scale.

SE sizes (e.g., radius 5 or 9 pixels) define the scale of depressions that will be filled during closing.

* Disk approximates isotropic cavities (common for pocket mouths).

* Diamond captures Manhattan geometry and can be less smoothing in corners.

* Square provides axis-aligned sensitivity and may segment elongated grooves differently.

Multiple sizes allow detection of pockets at different spatial scales.

In [ ]:
SEs = [
    ("disk", 5, disk(5)),
    ("disk", 9, disk(9)),
    ("diamond", 7, diamond(7)),
    ("square", 9, square(9)),
]


/tmp/ipython-input-2339581226.py:5: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  ("square", 9, square(9)),


### 4. Processing across all proteins and extracting region properties


Characterize and quantify each pocket candidate by size, shape, and depth statistics.

* area (pixels): size of the pocket region in the 2D projection

* eccentricity: elongation of the region (0 ≈ circular, 1 ≈ highly elongated)

* solidity: area divided by convex hull area (proxy for convexity/compactness)

* mean pocket depth: mean of pocket depth map inside the region (average depression strength)

* max pocket depth: maximum of pocket depth map inside the region (deepest point)

* polarity:"as_is" or "inverted"-because the meaning of bright vs dark in the depth PNG can depend on the projection convention, it tests both and selects the polarity that yields the most plausible pocket segmentation under the scoring criterion.

* score: D_max-λN (where D_max= max pocket depth, λ= penalty weight for balancing between more depth and more pockets, N= no. of pockets)


OUTPUTS:

1. Distance-to-boundary map: Estimates whether pockets lie near the projected outline (edge grooves) or more internally (potentially more “buried”)(in run_summary.csv — parameters tested per protein (SE type/size, polarity, threshold, pocket count, score))

2. Add mean distance-to-boundary per pocket. The mean of the distance transform within each pocket region. (saved in : pocket_descriptors.csv-one file per protein-one row for each pocket.)
3. A single figure per protein showing surface image before morphology, after closing, the pocket depth map (difference),
and the segmented pockets as boundaries.

4. Global summary table for cross-protein comparison for easy comparison between enzyme vs non-enzyme groups. ( saved in: all_proteins_summary.csv)

In [ ]:
all_summary = []

for depth_png in sorted(depth_dir.glob("*_depth.png")):
    pdb_id = depth_png.stem.replace("_depth", "")
    mask_png = mask_dir / f"{pdb_id}_mask.png"
    if not mask_png.exists():
        continue
    print(f"Processing {pdb_id}")
    # output dir for this protein
    out_dir = out_path / pdb_id
    out_dir.mkdir(parents=True, exist_ok=True)
    # load PNGs
    I0 = imread(depth_png).astype(float)
    if I0.ndim == 3: I0 = I0[...,0]
    I0 /= 255.0

    mask = imread(mask_png)
    if mask.ndim == 3: mask = mask[...,0]
    mask = mask > 127
    # clean mask
    mask = binary_closing(mask, iterations=2)
    mask = binary_fill_holes(mask)
    mask = keep_largest_component(mask)
    runs = []

    for polarity, I in [("as_is", I0), ("inverted", 1.0 - I0)]:
        for name, r, se in SEs:
            closed, pdepth, pbin, lbl, th = detect_pockets(I, mask, se)
            score = pdepth.max() - 0.06 * lbl.max()
            runs.append({
                "polarity": polarity,
                "se_shape": name,
                "se_size": r,
                "threshold": th,
                "n_pockets": int(lbl.max()),
                "max_depth": float(pdepth.max()),
                "score": score,
                "I": I,
                "closed": closed,
                "pdepth": pdepth,
                "lbl": lbl
            })

    runs_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ["I","closed","pdepth","lbl"]} for r in runs])
    runs_df = runs_df.sort_values("score", ascending=False)
    runs_df.to_csv(out_dir / "run_summary.csv", index=False)
    best = runs[runs_df.index[0]]

    # descriptors
    boundary = find_boundaries(mask, mode="outer")
    dist = distance_transform_edt(~boundary)
    props = regionprops_table(
        best["lbl"],
        intensity_image=best["pdepth"],
        properties=["label","area","eccentricity","solidity","mean_intensity","max_intensity"]
    )
    desc = pd.DataFrame(props).rename(columns={
        "mean_intensity":"mean_depth",
        "max_intensity":"max_depth"
    })
    if not desc.empty:
        desc["mean_dist_to_boundary_px"] = [
            dist[best["lbl"]==rid].mean() for rid in desc["label"]
        ]
    desc.to_csv(out_dir / "pocket_descriptors.csv", index=False)

    overlay = np.dstack([best["I"]]*3)
    overlay[find_boundaries(best["lbl"])] = [1,0,0]

    fig, ax = plt.subplots(1,4, figsize=(16,4))
    ax[0].imshow(best["I"], cmap="gray"); ax[0].set_title("Before")
    ax[1].imshow(best["closed"], cmap="gray"); ax[1].set_title("After closing")
    ax[2].imshow(best["pdepth"], cmap="gray"); ax[2].set_title("Closed − Before")
    ax[3].imshow(overlay); ax[3].set_title(f"Pockets (n={best['lbl'].max()})")
    for a in ax: a.axis("off")
    plt.tight_layout()
    plt.savefig(out_dir / "best_pockets.png", dpi=200)
    plt.close()

    all_summary.append({
        "pdb_id": pdb_id,
        "n_pockets": int(best["lbl"].max()),
        "max_depth": float(best["pdepth"].max()),
        "se_shape": best["se_shape"],
        "se_size": best["se_size"],
        "polarity": best["polarity"]
    })

pd.DataFrame(all_summary).to_csv(
    out_path / "all_proteins_summary.csv", index=False
)


Processing 1A3N
Processing 1AO6
Processing 1CA2
Processing 1LYZ
Processing 1MBN
Processing 2PTN





Discussion:

Although enzymes were expected to exhibit fewer pockets than non-enzymatic proteins, the morphology-based pipeline detected multiple pocket-like regions in both classes.

This outcome reflects the fact that the method identifies geometric surface depressions rather than biochemical active sites.

Also, the use of a 2D projection and quantized depth images limits discrimination based on pocket count alone. Nevertheless, enzymatic proteins tend to display a small number of dominant pockets with greater depth and spatial coherence

##Step 3: Data Analysis and Visualization
 *Tanushka Mantri*

---


###Goal

This step aimed to see whether enzyme proteins had distinct pocket properties in contrast to non-enzymatic proteins and to statistically assess the pocket descriptors produced by the morphological pipeline.

Geometric depressions were found and measured in Step 2, but Step 3 is concerned with adding together measures at the pocket level,
calculating summary statistics at the protein level, enzyme versus non-enzyme group comparison and conducting statistical analysis.




### 1. Dataset creation

All pocket_descriptors.csv files generated in Step 2 were loaded and concatenated into a single dataframe containing:

* area
* mean_depth
* max_depth
* eccentricity
* solidity
* mean_dist_to_boundary_px
* pdb_id


### 2.Feature extraction of each protein

Pocket-level descriptors were summarized per protein using biologically meaningful statistics:

* Number of pockets
* Median pocket area
* Median pocket depth
* Mean depth of the top 5 deepest pockets
* Maximum depth among the top 5 deepest pockets
* Mean distance to boundary
* Fraction of pockets far from the protein boundary

The “top 5” depth metrics were included to emphasize dominant pockets rather than noise from many shallow depressions.

---

In [ ]:

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu


results_dir = "/content/drive/MyDrive/BioimagingProject/results"
plots_dir = os.path.join(results_dir, "plots")

os.makedirs(plots_dir, exist_ok=True)

print("Results directory created at:", results_dir)
print("Plots will be saved to:", plots_dir)


file_paths = glob.glob("/content/drive/MyDrive/BioimagingProject/morphology/*/pocket_descriptors.csv")

all_data = []
for f in file_paths:
    df = pd.read_csv(f)
    pdb_id = os.path.basename(os.path.dirname(f))
    df["pdb_id"] = pdb_id
    all_data.append(df)

pockets = pd.concat(all_data, ignore_index=True)

print("Total pockets loaded:", len(pockets))


enzymes = ["1LYZ", "2PTN", "1CA2"]
pockets["is_enzyme"] = pockets["pdb_id"].isin(enzymes)
pockets.to_csv(os.path.join(results_dir, "all_pocket_data.csv"), index=False)


Results directory created at: /content/drive/MyDrive/BioimagingProject/results
Plots will be saved to: /content/drive/MyDrive/BioimagingProject/results/plots
Total pockets loaded: 116






### 4. Boundary Analysis

To evaluate whether pockets were surface grooves or potentially buried active sites, a distance threshold (median across all pockets) was used to classify pockets as:

* **Near boundary**
* **Far from boundary**

The fraction of boundary-distant pockets was calculated per protein.














In [ ]:

def top5_mean(series):
    return series.sort_values(ascending=False).head(5).mean()

def top5_max(series):
    return series.sort_values(ascending=False).head(5).max()

protein_stats = pockets.groupby("pdb_id").agg(
    n_pockets=("area", "count"),
    median_area=("area", "median"),
    median_depth=("mean_depth", "median"),
    top5_mean_depth=("mean_depth", top5_mean),
    top5_max_depth=("max_depth", top5_max),
    mean_dist_boundary=("mean_dist_to_boundary_px", "mean")
).reset_index()

protein_stats["is_enzyme"] = protein_stats["pdb_id"].isin(enzymes)

protein_stats.to_csv(
    os.path.join(results_dir, "protein_level_statistics.csv"),
    index=False
)

#Fraction of pockets far from boundary

distance_threshold = pockets["mean_dist_to_boundary_px"].median()
pockets["far_from_boundary"] = pockets["mean_dist_to_boundary_px"] > distance_threshold

fraction_far = (
    pockets.groupby("pdb_id")["far_from_boundary"]
    .mean()
    .reset_index(name="fraction_far")
)

protein_stats = protein_stats.merge(fraction_far, on="pdb_id")

protein_stats.to_csv(
    os.path.join(results_dir, "protein_level_statistics_with_boundary.csv"),
    index=False
)





### 5. Statistical Testing

For each protein-level feature, enzyme and non-enzyme groups were compared using the **Mann–Whitney U test**, a non-parametric statistical test appropriate for small sample sizes.

In [ ]:
group_summary = protein_stats.groupby("is_enzyme").agg(
    avg_n_pockets=("n_pockets", "mean"),
    avg_median_area=("median_area", "mean"),
    avg_median_depth=("median_depth", "mean"),
    avg_top5_mean_depth=("top5_mean_depth", "mean"),
    avg_top5_max_depth=("top5_max_depth", "mean"),
    avg_fraction_far=("fraction_far", "mean")
).reset_index()

group_summary["Group"] = group_summary["is_enzyme"].map(
    {True: "Enzyme", False: "Non-enzyme"}
)

group_summary.to_csv(
    os.path.join(results_dir, "group_level_summary.csv"),
    index=False
)

#enzyme vs non-enzyme
features = [
    "n_pockets", "median_area", "median_depth",
    "top5_mean_depth", "top5_max_depth", "fraction_far"
]

stat_results = []

for feature in features:
    enz = protein_stats[protein_stats["is_enzyme"]][feature]
    non = protein_stats[~protein_stats["is_enzyme"]][feature]

    try:
        stat, p = mannwhitneyu(enz, non, alternative="two-sided")
    except:
        stat, p = np.nan, np.nan

    stat_results.append({
        "Feature": feature,
        "Enzyme_Mean": enz.mean(),
        "NonEnzyme_Mean": non.mean(),
        "U_statistic": stat,
        "p_value": p
    })

stats_table = pd.DataFrame(stat_results)

stats_table.to_csv(
    os.path.join(results_dir, "statistical_comparison.csv"),
    index=False
)


### 6. Visualization


1. Histogram of pocket area distributions
2. Histogram of pocket depth distributions
3. Scatter plot of pocket area vs depth (pocket-level)
4. Scatter plot of number of pockets vs median depth (protein-level)

Color:

* Orange = Enzymes
* Blue = Non-enzymes



In [ ]:
ENZYME_COLOR = "#E07B54"
NON_ENZYME_COLOR = "#5B8DB8"


plt.figure()
plt.hist(pockets[pockets["is_enzyme"]]["area"], bins=20, alpha=0.6,
         color=ENZYME_COLOR, label="Enzymes")
plt.hist(pockets[~pockets["is_enzyme"]]["area"], bins=20, alpha=0.6,
         color=NON_ENZYME_COLOR, label="Non-enzymes")
plt.xlabel("Pocket area")
plt.ylabel("Count")
plt.legend()
plt.title("Pocket Area Distribution")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "hist_area.png"), dpi=300)
plt.close()


plt.figure()
plt.hist(pockets[pockets["is_enzyme"]]["mean_depth"], bins=20, alpha=0.6,
         color=ENZYME_COLOR, label="Enzymes")
plt.hist(pockets[~pockets["is_enzyme"]]["mean_depth"], bins=20, alpha=0.6,
         color=NON_ENZYME_COLOR, label="Non-enzymes")
plt.xlabel("Mean pocket depth")
plt.ylabel("Count")
plt.legend()
plt.title("Pocket Depth Distribution")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "hist_depth.png"), dpi=300)
plt.close()

# Scatter: Area vs Depth
plt.figure()
for label, group in pockets.groupby("is_enzyme"):
    plt.scatter(group["area"], group["mean_depth"],
                color=ENZYME_COLOR if label else NON_ENZYME_COLOR,
                label="Enzymes" if label else "Non-enzymes",
                alpha=0.7)
plt.xlabel("Pocket area")
plt.ylabel("Mean depth")
plt.legend()
plt.title("Pocket Area vs Depth")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "scatter_area_depth.png"), dpi=300)
plt.close()

# Protein-Level dot plot
import numpy as np
import matplotlib.pyplot as plt
import os

plt.figure()

for is_enz, grp in protein_stats.groupby("is_enzyme"):
    x0 = 1 if is_enz else 0
    offsets = np.linspace(-0.06, 0.06, len(grp))
    xs = x0 + offsets

    plt.scatter(
        xs,
        grp["median_depth"],
        color=ENZYME_COLOR if is_enz else NON_ENZYME_COLOR,
        s=140,
        alpha=0.85,
        label="Enzymes" if is_enz else "Non-enzymes"
    )

    for x, (_, r) in zip(xs, grp.iterrows()):
        plt.annotate(
            r["pdb_id"],
            (x, r["median_depth"]),
            textcoords="offset points",
            xytext=(6, 4),
            ha="left",
            fontsize=9
        )

    y_med = grp["median_depth"].median()
    plt.plot([x0 - 0.12, x0 + 0.12], [y_med, y_med], color="black", linewidth=2)

plt.xticks([0, 1], ["Non-enzymes", "Enzymes"])
plt.ylabel("Median pocket depth")
plt.title("Protein-Level Median Depth by Group")
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True, framealpha=0.85)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.grid(axis="y", alpha=0.25)
plt.xlim(-0.35, 1.35)
plt.ylim(0.12, 0.22)
plt.savefig(os.path.join(plots_dir, "protein_level_median_depth_dotplot.png"), dpi=300, bbox_inches="tight")
plt.close()


###Plot interpretations

1. Pocket Depth Distribution (Histogram):
There is a lot of overlap between the average pocket depth of enzymes and non-enzymes, with the majority of values falling within a similar moderate range, no obvious distinction.

2. Protein-Level Median Depth by Group (Dot plot):
At the protein level, median pocket depth varies across individual proteins in both groups. With only three proteins per group, there is no clear separation between enzymes and non-enzymes, and the plot mainly highlights protein-to-protein variability rather than a consistent group difference.

3. Pocket Area vs Depth (Scatter Plot):
There is no obvious correlation between pocket area and depth in either category, and both enzymes and non-enzymes follow similar patterns of clustering based on area ranges.

4. Pocket Area Distribution (Histogram):
Both categories have right-skewed distributions of pocket area with a majority of small-to-moderate pockets and a few large outliers, again suggesting that pocket area does not serve as a means of distinguishing functional proteins.

## Overall results

*Prepared by the full team*

---

This study tested whether simple morphological image processing (grayscale closing and hole filling applied to 2D projections of protein structures) can detect and measure candidate surface depressions (pockets) in projected depth maps.

### What worked

1. The morphology pipeline detected and segmented pocket-like regions for all proteins in the dataset.

2. For each protein, the pipeline extracted the same set of measurements for each pocket, including pocket count, area, depth, and distance to the projection boundary. These features were then compared between enzymes and non-enzymes.

### What we observed in the features

1.  Pocket count <br>
    Enzymes had a lower mean number of detected pockets (18.0) than non-enzymes (20.67). This is consistent with the idea that enzymatic proteins may have fewer dominant functional regions, while non-enzymes may show more incidental surface depressions. This trend is based on a small sample and should be treated as exploratory.

2. Depth-related metrics <br>
    Depth metrics did not show a clear separation between enzymes and non-enzymes. Median depth and top-5 depth summaries were slightly higher in non-enzymes on average, and the group differences were not statistically significant. This suggests that, with 2D projections and the current settings, pocket depth alone is not enzyme-specific.

      | Metric                     | Enzymes | Non-Enzymes | p-value |
      | ---------------------------| ------- | ----------- | ------- |
      | Median Pocket Depth        | 0.164   | 0.180       | 1.00    |
      | Top-5 Mean Depth           | 0.232   | 0.244       | 1.00    |
      | Top-5 Maximum Depth        | 0.739   | 0.924       | 0.184   |
      | Fraction Far from Boundary | 0.534   | 0.450       | 0.70    |


3. Distance from the projected boundary <br>
    Enzymes had a higher fraction of pockets farther from the 2D projection boundary (0.534 vs 0.450). Since functional sites are often partly buried rather than located at exposed edges, this directional trend is biologically plausible. However, it depends on how the cutoff for “far” is chosen and needs confirmation on more proteins.

4. Protein-level examples <br>
    Individual enzyme behavior further strengthens interpretation. For example, certain enzymes showed fewer but more structured depressions or higher localization. These protein-level patterns reinforce that morphology-based detection is not random but structurally informative.

    - 1CA2 (enzyme) had the highest mean area and largest mean boundary distance and showed very high internal pocket fraction (0.94)

    - 1AO6 (non-enzyme) had the highest pocket count (28).

    - 1LYZ (enzyme) had the fewest pockets (13) and showed mostly boundary-proximal pockets (0.08)


5. Statistical context and limitations <br>
    Mann–Whitney U tests did not find significant differences between enzymes and non-enzymes (p > 0.05). This is likely due to:
    - Small sample size (3 proteins per group)
    - Low statistical power at low n
    - Natural structural variability across proteins
    - Use of 2D projections, which compress 3D geometry

    Despite these limits, the pipeline provides a simple and interpretable way to detect and quantify pocket-like concavities from projected depth maps.

## Conclusion:

Simple morphological operations can reliably detect and quantify candidate pocket regions in 2D protein projections. In this small dataset, pocket count and boundary-location features showed directional trends, while depth metrics did not clearly separate enzymes from non-enzymes. Larger datasets and 3D-aware representations would be needed to test whether these measurements can support active-site detection.